# pdf to xlsx table converter for Shamiri academic data
Batch converts tabular PDF files to `.xlsx` workbooks.

### What this notebook does
1. Scans a local `data/` folder for all `.pdf` files.
2. Opens each PDF with `pdfplumber` and extracts all tables per page using `page.extract_tables()`.
3. Stacks all page-level tables into a single table per PDF, using the first extracted row as the column header.
4. Exports each PDF's combined table to `data/Exports/<original_filename>.xlsx`.

Forms stage 1 of the academic data QA


In [10]:
# importing libraries
import pandas as pd
from pathlib import Path
import re
import pdfplumber

In [11]:
# config
data_directory = Path.cwd().parent/'data'
input_folder = data_directory/'Inputs'
output_folder = input_folder

# find datasets in pdf format
datasets = [
    file
    for file in data_directory.rglob("*")
    if file.is_file()
    and file.suffix.lower() == '.pdf'
]

print(f"Found {len(datasets)} datasets:\n")

for dataset in datasets:
    print(dataset)

Found 9 datasets:

D:\imma\Automation\Shamiri\data\Inputs\AHERO_Grade 10- END Term 1 2026.pdf
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 3 - End term assessment - (2026 Term 1)_1783411946378.pdf
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 3 - Midterm - (2026 Term 2)_1782309390776.pdf_1782309390837.pdf
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 4 - End term assessment - (2026 Term 1)_1783504752376.pdf
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 4 - Midterm - (2026 Term 2)_1782309467394.pdf_1782309467450.pdf
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Grade 10 - End term assessment - (2026 Term 1)_1783411872536.pdf
D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Grade 10 - Midterm - (2026 Term 2)_1782309323652.pdf_1782309323684.pdf
D:\imma\Automation\Shamiri\data\Inputs\BOOGS_Form 3 - Module One Examination - (2026 Term 2)_1783084522091.pdf
D:\imma\Automation\Shamiri\data\Inputs\SILMS_ Grade 10 End 

In [12]:
# Extract all tables from PDF and classify them
def extract_pdf(pdf_file):
    academic_tables = []
    grade_breakdown_tables = []
    class_summary_tables = []

    with pdfplumber.open(pdf_file) as pdf:

        for page_no, page in enumerate(pdf.pages, start=1):

            # extract all tables on a page
            page_tables = page.extract_tables()

            for table_no, table in enumerate(page_tables, start=1):

                if not table:
                    continue

                # Remove completely empty rows
                table = [
                    row for row in table
                    if any(
                        cell is not None and str(cell).strip() != ""
                        for cell in row
                    )
                ]

                if not table:
                    continue

                # Clean cell values
                table = [
                    [
                        str(cell).strip() if cell is not None else ""
                        for cell in row
                    ]
                    for row in table
                ]

                # Identify the type of table
                first_rows_text = " ".join(
                    " ".join(row)
                    for row in table[:3]
                ).upper()

                # grade breakdown
                if "GRADE BREAKDOWN" in first_rows_text:

                    grade_breakdown_tables.append({
                        "page": page_no,
                        "table": table_no,
                        "data": table
                    })

                # class grade summary
                elif "CLASS GRADE SUMMARY" in first_rows_text:

                    class_summary_tables.append({
                        "page": page_no,
                        "table": table_no,
                        "data": table
                    })

                # If the title is not captured inside the table,
                # identify the table using its first column/header.
                else:

                    header_text = " ".join(table[0]).upper()

                    if "FORM" in header_text and (
                        "EE1" in header_text or
                        "EE2" in header_text or
                        "ME1" in header_text
                    ):
                        grade_breakdown_tables.append({
                            "page": page_no,
                            "table": table_no,
                            "data": table
                        })

                    elif "SUBJECT" in header_text and (
                        "EE1" in header_text or
                        "EE2" in header_text or
                        "ME1" in header_text
                    ):
                        class_summary_tables.append({
                            "page": page_no,
                            "table": table_no,
                            "data": table
                        })

                    else:
                        # Anything moved to student academic data table
                        academic_tables.append({
                            "page": page_no,
                            "table": table_no,
                            "data": table
                        })

    # Convert the academic tables into one DataFrame

    academic_df = None

    if academic_tables:

        # First academic table establishes the column names
        first_table = academic_tables[0]["data"]

        academic_header = first_table[1]

        academic_rows = []

        for item in academic_tables:

            table = item["data"]

            # Skip the first row because it is the header
            rows = table[1:]

            for row in rows:

                # Skip empty rows
                if not any(str(cell).strip() for cell in row):
                    continue

                # Remove repeated headers
                cleaned_row = [
                    str(cell).strip() if cell is not None else ""
                    for cell in row
                ]

                if cleaned_row == academic_header:
                    continue

                # Only add rows that match the academic table structure
                if len(cleaned_row) == len(academic_header):
                    academic_rows.append(cleaned_row)

        academic_df = pd.DataFrame(
            academic_rows,
            columns=academic_header
        )

    # Helper function for summary tables
    def combine_summary_tables(table_list):

        if not table_list:
            return None

        all_rows = []

        header = None

        for item in table_list:

            table = item["data"]

            if not table:
                continue

            # Find actual column header.
            table_header_index = None

            for i, row in enumerate(table[:5]):

                row_text = " ".join(row).upper()

                if (
                    "EE1" in row_text
                    and "ME1" in row_text
                ):
                    table_header_index = i
                    break

            if table_header_index is None:
                table_header_index = 0

            current_header = table[table_header_index]

            if header is None:
                header = current_header

            # Add data rows
            for row in table[table_header_index + 1:]:

                if not any(str(cell).strip() for cell in row):
                    continue

                cleaned_row = [
                    str(cell).strip() if cell is not None else ""
                    for cell in row
                ]

                # Remove repeated headers
                if cleaned_row == header:
                    continue

                # Keep the table as-is.
                all_rows.append(cleaned_row)

        # determine maximum no of columns
        if not all_rows:
            return None

        max_columns = max(
            len(row)
            for row in all_rows
        )

        # pad within this summary table df
        padded_rows = [
            row + [""] * (max_columns - len(row))
            for row in all_rows
        ]

        padded_header = (
            header +
            [""] * (max_columns - len(header))
        )

        return pd.DataFrame(
            padded_rows,
            columns=padded_header
        )

    # Combine summary tables
    grade_breakdown_df = combine_summary_tables(
        grade_breakdown_tables
    )

    class_summary_df = combine_summary_tables(
        class_summary_tables
    )

    return (
        academic_df,
        grade_breakdown_df,
        class_summary_df
    )

In [13]:
# Extract and export all datasets
for dataset in datasets:

    try:
        # extract pdf
        (
            academic_df,
            grade_breakdown_df,
            class_summary_df
        ) = extract_pdf(dataset)

        # Create output filename
        output_name = dataset.stem + ".xlsx"
        output_path = output_folder / output_name

        # Write to Excel
        with pd.ExcelWriter(
            output_path,
            engine="openpyxl"
        ) as writer:

            # Academic/student data
            if academic_df is not None and not academic_df.empty:

                academic_df.to_excel(
                    writer,
                    sheet_name="Academic_Data",
                    index=False
                )

            # Grade breakdown
            if (
                grade_breakdown_df is not None
                and not grade_breakdown_df.empty
            ):

                grade_breakdown_df.to_excel(
                    writer,
                    sheet_name="Grade_Breakdown",
                    index=False
                )

            # Class grade summary
            if (
                class_summary_df is not None
                and not class_summary_df.empty
            ):

                class_summary_df.to_excel(
                    writer,
                    sheet_name="Class_Grade_Summary",
                    index=False
                )

        # track exports
        print(f"\nExported: {output_path}")

        if academic_df is not None:
            print(
                f"  Academic_Data: "
                f"{len(academic_df)} rows"
            )

        if grade_breakdown_df is not None:
            print(
                f"  Grade_Breakdown: "
                f"{len(grade_breakdown_df)} rows"
            )

        if class_summary_df is not None:
            print(
                f"  Class_Grade_Summary: "
                f"{len(class_summary_df)} rows"
            )

    except Exception as e:

        print(f"\nFailed to export: {dataset}")
        print(f"Error: {e}")


Exported: D:\imma\Automation\Shamiri\data\Inputs\AHERO_Grade 10- END Term 1 2026.xlsx
  Academic_Data: 649 rows
  Grade_Breakdown: 13 rows
  Class_Grade_Summary: 22 rows

Exported: D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 3 - End term assessment - (2026 Term 1)_1783411946378.xlsx
  Academic_Data: 76 rows
  Grade_Breakdown: 4 rows
  Class_Grade_Summary: 13 rows

Exported: D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 3 - Midterm - (2026 Term 2)_1782309390776.pdf_1782309390837.xlsx
  Academic_Data: 76 rows
  Grade_Breakdown: 4 rows
  Class_Grade_Summary: 13 rows

Exported: D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 4 - End term assessment - (2026 Term 1)_1783504752376.xlsx
  Academic_Data: 83 rows
  Grade_Breakdown: 4 rows
  Class_Grade_Summary: 13 rows

Exported: D:\imma\Automation\Shamiri\data\Inputs\ALARA_Merit-List_Form 4 - Midterm - (2026 Term 2)_1782309467394.pdf_1782309467450.xlsx
  Academic_Data: 83 rows
  Grade_Breakdown: 4 r